# 👟🩴 ShoeMatch AI — Colab Training & Dataset Indexer
### Fast GPU-Accelerated Embedding Extraction & FAISS Index Builder with Automatic Shoe vs. Slipper Tagging

This notebook allows you to upload your custom shoe & slipper dataset, compute deep metric embeddings using Meta **DINOv2** / **CLIP** on free Google Colab GPUs, and export a ready-to-use package (`shoe_matching_colab_export.zip`) that plugs directly into your local or cloud deployment.

---

### 📁 Expected Dataset Folder Structure:
You can organize your dataset in either of these two formats:

**Format A (Categorized Folders - Recommended):**
```
dataset/
  ├── shoes/
  │     ├── design_001_airmax_style/
  │     │     ├── angle_side.jpg
  │     │     └── angle_top.jpg
  │     └── design_002_classic_loafer/
  │           └── photo_1.jpg
  └── slippers/
        ├── design_101_flip_flop_classic/
        │     └── img1.jpg
        └── design_102_fuzzy_home_slipper/
              └── img1.jpg
```

**Format B (Flat Design Folders):**
```
dataset/
  ├── design_001_sneaker/
  │     └── img1.jpg
  └── design_002_slipper_slide/
        └── img1.jpg
```

## 1. Install Dependencies

In [ ]:
!pip install -q torch torchvision transformers faiss-cpu pillow numpy open-clip-torch sentence-transformers

## 2. Upload and Extract Dataset Zip
Run this cell and select your dataset `.zip` file from your computer.

In [ ]:
import os, zipfile, shutil
from google.colab import files

print("Please upload your dataset zip file (e.g., dataset.zip):")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
os.makedirs("raw_dataset", exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall("raw_dataset")

print(f"Extracted {zip_name} successfully!")

## 3. GPU Embedding Extraction & FAISS Index Construction

In [ ]:
import os, glob, json, time
from pathlib import Path
import numpy as np
from PIL import Image, ImageOps
import torch
import torch.nn.functional as F
import faiss
from transformers import AutoImageProcessor, AutoModel

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

# Load Vision Backbone (DINOv2)
MODEL_NAME = "facebook/dinov2-small"
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

# Helper: Preprocess image
def load_image(img_path):
    img = Image.open(img_path)
    img = ImageOps.exif_transpose(img)
    if img.mode != "RGB":
        img = img.convert("RGB")
    return img

# Helper: Compute L2-normalized embedding
def get_embedding(img):
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        features = outputs.last_hidden_state[:, 0, :]
        features = F.normalize(features, p=2, dim=-1)
    return features.squeeze(0).cpu().numpy().astype(np.float32)

# Scan extracted dataset
export_dir = Path("colab_export")
export_images_dir = export_dir / "catalog_images"
export_dir.mkdir(parents=True, exist_ok=True)
export_images_dir.mkdir(parents=True, exist_ok=True)

all_image_paths = []
for ext in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
    all_image_paths.extend(glob.glob(f"raw_dataset/**/{ext}", recursive=True))

print(f"Found {len(all_image_paths)} total images to process.")

# Group images by design folder
designs_map = {}
for p in all_image_paths:
    path_obj = Path(p)
    folder_name = path_obj.parent.name
    grandparent = path_obj.parent.parent.name.lower()
    
    # Determine category (shoe vs slipper)
    if "slipper" in grandparent or "slipper" in folder_name.lower() or "slide" in folder_name.lower() or "flip" in folder_name.lower():
        category = "slipper"
    else:
        category = "shoe"
        
    if folder_name not in designs_map:
        designs_map[folder_name] = {"category": category, "images": []}
    designs_map[folder_name]["images"].append(p)

print(f"Identified {len(designs_map)} unique footwear designs.")

# Indexing loop
embeddings = []
metadata_records = []
faiss_id = 0
t0 = time.time()

for d_idx, (folder_name, data) in enumerate(designs_map.items(), start=1):
    cat_type = data["category"]
    prefix = "SLIP" if cat_type == "slipper" else "SHOE"
    design_id = f"{prefix}-{d_idx:03d}"
    design_name = folder_name.replace("_", " ").replace("-", " ").title()
    
    target_folder = export_images_dir / design_id
    target_folder.mkdir(parents=True, exist_ok=True)
    
    for img_idx, img_p in enumerate(data["images"], start=1):
        img_name = f"photo_{img_idx}.jpg"
        dest_path = target_folder / img_name
        
        # Load and compute embedding
        pil_img = load_image(img_p)
        pil_img.save(dest_path, "JPEG", quality=92)
        
        emb = get_embedding(pil_img)
        embeddings.append(emb)
        
        metadata_records.append({
            "faiss_id": faiss_id,
            "design_id": design_id,
            "name": design_name,
            "category": cat_type,
            "category_normalized": cat_type,
            "angle": "side" if img_idx == 1 else f"angle_{img_idx}",
            "image_path": f"/catalog_images/{design_id}/{img_name}",
            "shelf_location": f"Warehouse {'B' if cat_type == 'slipper' else 'A'} - Rack {d_idx%10 + 1:02d} - Shelf {d_idx%4 + 1}",
            "materials": "EVA Foam / Soft Strap" if cat_type == "slipper" else "Full Grain Leather / Rubber Sole",
            "season": "Collection 2026"
        })
        faiss_id += 1

# Build FAISS Index
emb_matrix = np.vstack(embeddings).astype(np.float32)
dim = emb_matrix.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(emb_matrix)

# Save FAISS index and metadata.json
faiss_file_path = export_dir / "shoe_index.faiss"
faiss.write_index(index, str(faiss_file_path))

meta_file_path = export_dir / "metadata.json"
with open(meta_file_path, "w", encoding="utf-8") as f:
    json.dump(metadata_records, f, indent=2)

print(f"\n Successfully indexed {len(metadata_records)} images across {len(designs_map)} designs in {time.time()-t0:.2f}s!")
print(f"FAISS Index Vectors: {index.ntotal} (Dim: {dim})")

## 4. Package & Download Export Zip
This creates `shoe_matching_colab_export.zip` and automatically downloads it to your computer.

In [ ]:
import shutil
from google.colab import files

zip_export_path = shutil.make_archive("shoe_matching_colab_export", "zip", "colab_export")
print(f"Export zip generated: {zip_export_path}")
print("Downloading package now...")
files.download(zip_export_path)

## 5. How to Load in Your Project Locally:
1. Move `shoe_matching_colab_export.zip` to your project folder.
2. Run:
```bash
python import_colab_index.py --zip shoe_matching_colab_export.zip
```
3. Restart the server (`python run_server.py`) and open `http://localhost:8000`!